# 2016〜2026年 衆議院予算委員会 発言トピック・クラスタリング

11暦年のクリーニング済み68,948発言を共通の特徴空間で10クラスタに分類します。2026年は取得時点までの途中年です。

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'cleaned').exists():
    ROOT = ROOT.parent
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(ROOT))
from clustering_2016 import (
    FIXED_K, RANDOM_STATE, apply_manual_labels, cluster_details, compare_k,
    export_results, load_input, reduce_dimensions, vectorize,
)

sns.set_theme(style='whitegrid')
mac_font_paths = list(Path('/System/Library/Fonts').glob('*角*W3.ttc'))
if not mac_font_paths:
    raise FileNotFoundError('macOSのヒラギノ角ゴシック W3が見つかりません')
font_manager.fontManager.addfont(str(mac_font_paths[0]))
MAC_JAPANESE_FONT = font_manager.FontProperties(fname=str(mac_font_paths[0])).get_name()
plt.rcParams['font.family'] = MAC_JAPANESE_FONT
plt.rcParams['axes.unicode_minus'] = False
OUTPUT_DIR = ROOT / 'analysis/clustering_2016_2026'
EXPECTED_BY_YEAR = {2016: 5928, 2017: 6283, 2018: 7434, 2019: 5985, 2020: 8056, 2021: 6269, 2022: 6451, 2023: 5744, 2024: 5896, 2025: 7449, 2026: 3453}
print(f'Matplotlib font: {MAC_JAPANESE_FONT} ({mac_font_paths[0]})')
RANDOM_STATE

## 1. 年別入力の読込・検証

各年の件数、会議名、発言ID、本文を検証してから縦結合します。

In [ ]:
frames = [
    load_input(ROOT / f'cleaned/speeches_{year}.jsonl', expected_rows=count)
    for year, count in EXPECTED_BY_YEAR.items()
]
df = pd.concat(frames, ignore_index=True, verify_integrity=True)
assert len(df) == 68948
assert df['speechID'].is_unique
assert set(df['year']) == set(EXPECTED_BY_YEAR)
year_counts = df.groupby('year', as_index=False).agg(speeches=('speechID', 'size'), speakers=('speaker', 'nunique'))
display(year_counts)
display(pd.Series({'rows': len(df), 'unique_speech_ids': df['speechID'].nunique(), 'years': df['year'].nunique(), 'review_flagged': int(df['review_flags'].map(bool).sum())}).to_frame('value'))

## 2. 日本語処理、TF-IDF、次元削減

全年度をまとめてSudachiPyで正規化し、共通TF-IDF空間を作成して100次元へ圧縮します。

In [ ]:
source_df = df.copy()
vectorizer, tfidf, keep_mask = vectorize(source_df)
zero_vector_records = source_df.loc[~keep_mask].copy()
zero_vector_records['exclusion_reason'] = 'zero_vector_after_tfidf'
df = source_df.loc[keep_mask].copy().reset_index(drop=True)
assert len(df) + len(zero_vector_records) == len(source_df)
assert not {'発言短文', '低情報発言'} & set(vectorizer.get_feature_names_out())
svd, reduced = reduce_dimensions(tfidf)
print(f'TF-IDF: {tfidf.shape[0]:,} rows × {tfidf.shape[1]:,} features')
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.1%}')
print(f'Zero vectors excluded: {len(zero_vector_records):,}')
display(zero_vector_records[['speechID', 'date', 'year', 'speaker', 'speech_cleaned', 'exclusion_reason']])

## 3. KMeans（10クラスタ固定）

2016〜2026年を一つのモデルで `k=10` に分類します。

In [ ]:
scores, models = compare_k(reduced, candidates=[FIXED_K])
best_k = FIXED_K
model = models[best_k]
print(f'Fixed k={best_k}')
display(scores.style.format({'silhouette': '{:.4f}', 'inertia': '{:,.1f}'}))

## 4. クラスタの解釈

特徴語、代表発言、主要話者、クラスタ規模を確認します。

In [ ]:
assignments, summary = cluster_details(df, tfidf, reduced, vectorizer, model)
display(summary.style.format({'ratio': '{:.1%}'}))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_sample = assignments.sample(n=min(10000, len(assignments)), random_state=RANDOM_STATE)
plot_idx = plot_sample.index.to_numpy()
sns.scatterplot(x=reduced[plot_idx, 0], y=reduced[plot_idx, 1], hue=plot_sample['cluster'], palette='tab10', s=10, alpha=.45, legend='full', ax=axes[0])
axes[0].set(title='SVD先頭2軸（固定標本10,000件）', xlabel='SVD 1', ylabel='SVD 2')
sns.barplot(data=summary, x='cluster', y='count', hue='cluster', palette='tab10', legend=False, ax=axes[1])
axes[1].set_title('クラスタ件数')
plt.tight_layout();

In [ ]:
for cluster_id in summary['cluster']:
    row = summary.loc[summary['cluster'].eq(cluster_id)].iloc[0]
    members = assignments.loc[assignments['cluster'].eq(cluster_id)]
    representatives = members.nsmallest(5, 'centroid_distance')[['speechID', 'date', 'year', 'speaker', 'speech_cleaned', 'centroid_distance']]
    top_speakers = members['speaker'].value_counts().head(5).rename_axis('speaker').reset_index(name='count')
    print(f"\nCluster {cluster_id}: {row['provisional_label']} ({row['count']:,}件)")
    print(f"特徴語: {row['top_terms']}")
    display(top_speakers, representatives)

## 5. 年別クラスタ構成

2026年は途中年なので、他年との比率比較ではその点に注意します。

In [ ]:
yearly_counts = pd.crosstab(assignments['year'], assignments['cluster'])
yearly_ratio = pd.crosstab(assignments['year'], assignments['cluster'], normalize='index')
display(yearly_counts)
plt.figure(figsize=(14, 6))
sns.heatmap(yearly_ratio.T, cmap='YlGnBu', annot=True, fmt='.0%')
plt.title('年ごとのクラスタ構成比（2026年は途中年）')
plt.xlabel('年')
plt.ylabel('クラスタ')
plt.tight_layout();

## 6. 人手によるラベル調整

代表発言を確認し、必要なクラスタだけ名称を上書きしてください。

In [ ]:
MANUAL_CLUSTER_LABELS = {
    # 0: '社会保障・医療',
    # 1: '外交・安全保障',
}
assignments, summary = apply_manual_labels(assignments, summary, MANUAL_CLUSTER_LABELS)
display(summary[['cluster', 'cluster_label', 'count', 'ratio', 'top_terms']].style.format({'ratio': '{:.1%}'}))

## 7. CSV出力と最終検証

発言別結果、クラスタ要約、年別件数・構成比をUTF-8 BOM付きで保存します。

In [ ]:
assert len(assignments) + len(zero_vector_records) == 68948
assert assignments['speechID'].is_unique
assert assignments['cluster'].notna().all()
assert set(assignments['cluster']) == set(range(10))
assert summary['count'].sum() == len(assignments)
export_results(assignments, summary, OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
zero_vector_records.to_csv(OUTPUT_DIR / 'excluded_zero_vectors.csv', index=False, encoding='utf-8-sig')
yearly_counts.to_csv(OUTPUT_DIR / 'yearly_cluster_counts.csv', encoding='utf-8-sig')
yearly_ratio.to_csv(OUTPUT_DIR / 'yearly_cluster_ratios.csv', encoding='utf-8-sig')
assert len(pd.read_csv(OUTPUT_DIR / 'speech_clusters.csv')) == len(assignments)
assert len(pd.read_csv(OUTPUT_DIR / 'excluded_zero_vectors.csv')) == len(zero_vector_records)
assert len(pd.read_csv(OUTPUT_DIR / 'cluster_summary.csv')) == 10
print(f'出力先: {OUTPUT_DIR}')
print(f'検証完了: 元{len(source_df):,}件、ゼロベクトル除外{len(zero_vector_records):,}件、学習対象{len(assignments):,}件、10クラスタ')